# Google Meridian MMM (example): panel from `MMM Data.csv`

This notebook is an **example / smoke test** only. It builds a **geo × time** table from `examples/data/MMM Data.csv` using **`dashboard_rmse_optimized.load_real_mmm_data`** (same code as the dashboard) plus [`pymc_marketing_mmm_pooled_dma.ipynb`](pymc_marketing_mmm_pooled_dma.ipynb)-style steps: `UnifiedDataPipeline.temporal_split`, `SimpleGlobalScaler`, DeepCausalMMM `examples/pymc_aligned_dcm_config.json`.

**Policy:** Do **not** edit `examples/dashboard_rmse_optimized.py` for this workflow; the notebook imports `load_real_mmm_data` from that file.

**Requirements:** **`google-meridian`** needs **Python ≥ 3.10** (current PyPI versions; 3.11+ recommended). Uses **TensorFlow / TFP** under the hood. Install: `pip install google-meridian`.

**Spend proxy:** The CSV has **impressions** only. Meridian expects **media + media_spend**. This example sets **`media_spend = impressions`** (documented proxy; replace with real spend for production).

**Holdout:** For **parity** with the PyMC notebook we use the **last `holdout_ratio` fraction of weeks × all geos**. Meridian’s docs recommend a **balanced** holdout and caution against large **contiguous tail** holdouts for knot-based trend/seasonality—see [Holdout observations](https://developers.google.com/meridian/docs/advanced-modeling/holdout-observations). Treat out-of-sample metrics here as **illustrative**, not best practice for Meridian.

**Speed:** Set `MAX_DMAS` (e.g. 20–50) and small MCMC `n_*` below for a quick demo.

**R² vs rapidMMM / other MMMs:** Meridian’s `predictive_accuracy` R² compares KPI to the **posterior mean expected outcome** (full model, including **geo effects** and **knot-based time** structure in geo models). That baseline is **much more flexible** than many custom or neural MMMs, so **Train R² can sit near 1.0** without implying the same incremental-media identification you see elsewhere. **Do not compare train R² across tools** as if they were the same estimand; align definitions or use **comparable holdout / CV** metrics instead.


## Configuration


In [6]:
import json
import os
import sys
from pathlib import Path


def _deepcausalmmm_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for d in [cwd, *cwd.parents]:
        if (d / "pyproject.toml").is_file() and (d / "deepcausalmmm").is_dir():
            return d
    raise FileNotFoundError(
        "Could not find deepcausalmmm repo root. cd to the repo or examples/."
    )


REPO_ROOT = _deepcausalmmm_repo_root()
_repo = str(REPO_ROOT)
if _repo not in sys.path:
    sys.path.insert(0, _repo)

_cfg_path = REPO_ROOT / "examples" / "pymc_aligned_dcm_config.json"
if not _cfg_path.is_file():
    raise FileNotFoundError(f"Missing {_cfg_path}")
with open(_cfg_path, encoding="utf-8") as f:
    DCM_CONFIG = json.load(f)

DATA_PATH = os.path.join("examples", "data", "MMM Data.csv")
if not os.path.isfile(DATA_PATH):
    DATA_PATH = os.path.join("data", "MMM Data.csv")

# Subset geos for a faster demo (None = all DMAs).
MAX_DMAS = 25

# Meridian MCMC smoke settings (increase for real use).
N_CHAINS = 1
N_ADAPT = 150
N_BURNIN = 100
N_KEEP = 50

HOLDOUT_RATIO = float(DCM_CONFIG.get("holdout_ratio", 0.12))
geo_col = "dmacode"

print(
    f"REPO_ROOT={REPO_ROOT} | holdout_ratio={HOLDOUT_RATIO} | MAX_DMAS={MAX_DMAS} | "
    f"MCMC chains={N_CHAINS} adapt={N_ADAPT} burnin={N_BURNIN} keep={N_KEEP}"
)


REPO_ROOT=/Users/adityapu/Documents/GitHub/deepcausalmmm | holdout_ratio=0.12 | MAX_DMAS=25 | MCMC chains=1 adapt=150 burnin=100 keep=50


## Imports


In [7]:
import importlib.util
import numpy as np
import pandas as pd
import sys

if sys.version_info < (3, 10):
    raise RuntimeError(
        "google-meridian requires Python 3.10+ (3.11+ recommended). "
        f"Current: {sys.version}"
    )

try:
    from meridian import constants as mer_constants
    from meridian.data.data_frame_input_data_builder import DataFrameInputDataBuilder
    from meridian.model.model import Meridian
    from meridian.model.spec import ModelSpec
    from meridian.analysis.analyzer import Analyzer
except ImportError as e:
    raise ImportError(
        "Install Meridian in this environment: pip install google-meridian\\n"
        f"Original error: {e}"
    ) from e

from deepcausalmmm.core.data import UnifiedDataPipeline
from deepcausalmmm.core.scaling import SimpleGlobalScaler


## Build panel → long `mer_df` (Meridian columns)


In [8]:
if not os.path.isfile(DATA_PATH):
    raise FileNotFoundError(f"MMM data not found: {DATA_PATH}")

# ----- Panel: same as dashboard — examples/dashboard_rmse_optimized.load_real_mmm_data -----
from pathlib import Path

_dash_py = REPO_ROOT / "examples" / "dashboard_rmse_optimized.py"
if not _dash_py.is_file():
    raise FileNotFoundError(f"Missing {_dash_py}")
_spec = importlib.util.spec_from_file_location("dashboard_rmse_optimized", _dash_py)
_dash = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_dash)

_data_abs = Path(DATA_PATH)
if not _data_abs.is_file():
    _data_abs = REPO_ROOT / DATA_PATH
if not _data_abs.is_file():
    _data_abs = REPO_ROOT / "data" / "MMM Data.csv"
if not _data_abs.is_file():
    raise FileNotFoundError(f"MMM data not found: {DATA_PATH}")

X_media, X_control, y, media_names, control_names = _dash.load_real_mmm_data(str(_data_abs))

hdr = pd.read_csv(_data_abs, nrows=0)
impression_cols = [c for c in hdr.columns if "impressions_" in c]
if "target_visits" in hdr.columns:
    target_col = "target_visits"
    value_cols = [c for c in hdr.columns if c.startswith("control_")]
else:
    target_col = "value_visits_visits"
    value_cols = [c for c in hdr.columns if "value_" in c and c != "value_visits_visits"]

regions = sorted(pd.read_csv(_data_abs, usecols=["dmacode"])["dmacode"].unique().tolist())
if MAX_DMAS is not None:
    regions = regions[: int(MAX_DMAS)]
    _k = len(regions)
    X_media, X_control, y = X_media[:_k], X_control[:_k], y[:_k]

n_weeks = int(X_media.shape[1])

pipeline = UnifiedDataPipeline(DCM_CONFIG)
train_data, holdout_data = pipeline.temporal_split(X_media, X_control, y)
train_weeks = int(train_data["X_media"].shape[1])
holdout_weeks = int(holdout_data["X_media"].shape[1])
assert train_weeks + holdout_weeks == n_weeks

scaler = SimpleGlobalScaler(DCM_CONFIG)
scaler.fit(train_data["X_media"], train_data["X_control"], train_data["y"])
_, X_ctrl_train_t, _ = scaler.transform(
    train_data["X_media"], train_data["X_control"], train_data["y"]
)
_, X_ctrl_hold_t, _ = scaler.transform(
    holdout_data["X_media"], holdout_data["X_control"], holdout_data["y"]
)
X_ctrl_train = X_ctrl_train_t.numpy()
X_ctrl_hold = X_ctrl_hold_t.numpy()

channel_short = [f"ch{i:02d}" for i in range(1, len(impression_cols) + 1)]
control_cols = list(value_cols)
dates_all = [pd.Timestamp("2020-01-06") + pd.Timedelta(weeks=i) for i in range(n_weeks)]

rows_tr, rows_ho = [], []
for ri, code in enumerate(regions):
    for ti in range(train_weeks):
        row = {geo_col: code, "date": dates_all[ti]}
        for c in range(len(channel_short)):
            row[channel_short[c]] = float(train_data["X_media"][ri, ti, c])
        for c in range(len(control_cols)):
            row[control_cols[c]] = float(X_ctrl_train[ri, ti, c])
        row[target_col] = float(train_data["y"][ri, ti])
        rows_tr.append(row)
    for ti in range(holdout_weeks):
        gti = train_weeks + ti
        row = {geo_col: code, "date": dates_all[gti]}
        for c in range(len(channel_short)):
            row[channel_short[c]] = float(holdout_data["X_media"][ri, ti, c])
        for c in range(len(control_cols)):
            row[control_cols[c]] = float(X_ctrl_hold[ri, ti, c])
        row[target_col] = float(holdout_data["y"][ri, ti])
        rows_ho.append(row)

df_train = pd.DataFrame(rows_tr)
df_hold = pd.DataFrame(rows_ho)
df_panel = pd.concat([df_train, df_hold], ignore_index=True)

# Meridian long table: string geo/time, kpi, controls, media + spend proxy
mer_df = df_panel.copy()
mer_df["geo"] = mer_df[geo_col].astype(str)
mer_df["time"] = pd.to_datetime(mer_df["date"]).dt.strftime("%Y-%m-%d")
mer_df["kpi"] = mer_df[target_col].astype(float)
mer_df["revenue_per_kpi"] = 1.0
mer_df["population"] = 1.0e6

spend_cols = []
for ch in channel_short:
    sc = f"spend_{ch}"
    mer_df[sc] = mer_df[ch].astype(float)
    spend_cols.append(sc)

print(
    f"Meridian input | geos={mer_df['geo'].nunique()} times={mer_df['time'].nunique()} | "
    f"train_weeks={train_weeks} holdout_weeks={holdout_weeks} | channels={len(channel_short)}"
)
mer_df[["geo", "time", "kpi"] + channel_short[:2]].head()


INFO - 
 Unified Data Pipeline - Time Series Split (Ratio-Based):
INFO -     Training: 96 actual weeks (weeks 1-96) - 88.1%
INFO -     Holdout: 13 actual weeks (weeks 97-109) - 11.9%
INFO -     Burn-in Padding: 6 weeks will be added to BOTH train and holdout
INFO -     Model sees: Train 102 weeks, Holdout 19 weeks
INFO -     Evaluation: Remove 6 padding weeks, evaluate on ALL actual data
INFO -     Time series approach: Training on historical data, testing on most recent data
Meridian input | geos=25 times=109 | train_weeks=96 holdout_weeks=13 | channels=13


,geo,time,kpi,ch01,ch02
0,500,2020-01-06,624179.0,812.0,24055.0
1,500,2020-01-13,637085.0,1212.0,25781.0
2,500,2020-01-20,641187.0,1066.0,25126.0
3,500,2020-01-27,627668.0,438.0,31069.0
4,500,2020-02-03,640611.0,406.0,27191.0


## Fit Meridian (smoke MCMC) and predictive accuracy


In [9]:
# holdout_id must match Meridian's (geo, time) axis order AFTER builder.build().
# sorted(mer_df['geo']) is lexicographic on string DMAs (e.g. '501' vs '5001') and
# does NOT match natsort inside the builder → wrong weeks marked holdout → bogus Test R².
import numpy as np
import time

pop_df = mer_df.groupby("geo", as_index=False)["population"].first()

media_channel_names = list(channel_short)
builder = (
    DataFrameInputDataBuilder(kpi_type=mer_constants.NON_REVENUE)
    .with_kpi(mer_df, kpi_col="kpi", time_col="time", geo_col="geo")
    .with_revenue_per_kpi(mer_df, revenue_per_kpi_col="revenue_per_kpi", time_col="time", geo_col="geo")
    .with_population(pop_df, population_col="population", geo_col="geo")
    .with_controls(mer_df, control_cols=control_cols, time_col="time", geo_col="geo")
    .with_media(
        mer_df,
        media_cols=channel_short,
        media_spend_cols=spend_cols,
        media_channels=media_channel_names,
        time_col="time",
        geo_col="geo",
    )
)
input_data = builder.build()

_gdim, _tdim = mer_constants.GEO, mer_constants.TIME
n_g = int(input_data.kpi.sizes[_gdim])
n_t = int(input_data.kpi.sizes[_tdim])
holdout_start = n_t - int(holdout_weeks)
holdout_id = np.zeros((n_g, n_t), dtype=bool)
holdout_id[:, holdout_start:] = True

model_spec = ModelSpec(holdout_id=holdout_id)
mmm = Meridian(input_data, model_spec)

t0 = time.perf_counter()
mmm.sample_posterior(
    n_chains=N_CHAINS,
    n_adapt=N_ADAPT,
    n_burnin=N_BURNIN,
    n_keep=N_KEEP,
    seed=42,
)
print(f"sample_posterior wall time: {time.perf_counter() - t0:.1f} s")

# Prefer model_context + inference_data (avoids meridian= deprecation warning).
analyzer = Analyzer(model_context=mmm.model_context, inference_data=mmm.inference_data)
acc = analyzer.predictive_accuracy(use_kpi=True)

# `print(acc)` on an xarray.Dataset truncates numbers as "0...." in stdout.
# Materialize a flat table so R² / MAPE / wMAPE are visible.
_metrics_df = acc.to_dataframe().reset_index()
print(_metrics_df.to_string(index=False))


/opt/homebrew/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/opt/homebrew/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/opt/homebrew/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/opt/homebrew/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
2026-03-21 20:55:20.086628: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator mcmc_retry_init/assert_equal_1/Assert/AssertGuard/Assert


sample_posterior wall time: 171.4 s
   metric geo_granularity evaluation_set        value
R_Squared             geo          Train     0.996865
R_Squared             geo           Test    -2.319076
R_Squared             geo       All Data     0.511461
R_Squared        national          Train     0.999198
R_Squared        national           Test -8745.105469
R_Squared        national       All Data   -96.110680
     MAPE             geo          Train          inf
     MAPE             geo           Test     9.446736
     MAPE             geo       All Data          inf
     MAPE        national          Train     0.002336
     MAPE        national           Test     2.231830
     MAPE        national       All Data     0.268239
    wMAPE             geo          Train     0.051067
    wMAPE             geo           Test     2.249846
    wMAPE             geo       All Data     0.345169
    wMAPE        national          Train     0.002256
    wMAPE        national           Test     2

/opt/homebrew/lib/python3.11/site-packages/arviz/data/inference_data.py:157: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/arviz/data/inference_data.py:1647: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/meridian/analysis/analyzer.py:676: RuntimeWarning: divide by zero encountered in divide
  return backend.nanmean(backend.absolute((actual - expected) / actual))


## Notes

- **`Analyzer.predictive_accuracy`** returns an **`xarray.Dataset`**. **`print(dataset)`** often shows `value ... 0....` — xarray is **truncating** the array in the text repr; metrics are still there. Use **`acc.to_dataframe()`** (as in the code cell) to print **R_Squared**, **MAPE**, and **wMAPE** by `geo_granularity` and `evaluation_set` (`Train` / `Test` / `All Data` when `holdout_id` is set). See Meridian docs.
- For **production**, replace the spend proxy, tune priors / `ModelSpec`, use **balanced** holdouts, and run **longer** MCMC.
- **Related:** [`pymc_marketing_mmm_pooled_dma.ipynb`](pymc_marketing_mmm_pooled_dma.ipynb).
